NER MODEL

MEDAES QA

In [9]:
# Script to extract questions from the 'medaesqa_v1.json' dataset
import json

def get_json():
    try:
        with open('datasets/medaesqa_v1.json', 'r', encoding="utf-8") as file:
            print("Successfully loaded 'medaesqa_v1.json'.")
            return json.load(file)
    except FileNotFoundError:
        print("File not found. Please ensure 'medaesqa_v1.json' is in the current directory.")
        return None
    except json.JSONDecodeError:
        print("Error decoding JSON. Please ensure 'medaesqa_v1.json' contains valid JSON.")
        return None
    

def extract_questions(data):
    questions = []

    if data:
        for item in data:
            if "question" in item:
                questions.append(item["question"])
    else:
        print("No data found.")

    return questions



data = get_json()
questions = extract_questions(data)

for q in questions:
    print(q)

Successfully loaded 'medaesqa_v1.json'.
Are there ways to prevent sleep apnea or treat it naturally?
What will mutation in runx2 affect in the future?
how long does a minor cornea injury take to heal without medical attention?
what drug or combination of drugs is most popular for treating stage 4 copd?
what causes left sided facial numbness?
how can i lower my cortisol level?
what is a t wave abnormality?
what is fbn1 mutation?
what does cad without angina mean?
what is tamsulosin used for?
what is nondisjunction and what causes it?
what effects does gene therapy have on an organism?
how does cbd effect liver enzymes?
what are side effects for suddenly stopping metoprolol?
is pcos linked to oxidative stress?
what are side effects of using formoterol?
what is the effect of low vitamin D?
What are the short and long term effects of surgery and its complexity for women with ovarian cyst?
what are the symptoms and treatments for ibs?
what can happen if you are prescribed thyroid meds but d

In [10]:
# Script to convert extracted questions to csv file
import csv
def save_to_csv(questions, filename='extracted_questions.csv'):
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Question'])  # Write header
            for question in questions:
                writer.writerow([question])  # Write each question in a new row
        print(f"Questions successfully saved to '{filename}'.")
    except Exception as e:
        print(f"Error occurred while saving to CSV: {e}")  
        
save_to_csv(questions)

Questions successfully saved to 'extracted_questions.csv'.


In [11]:
import re

def classify_medical_intent(query):
    query = query.lower()
    
    # Define keywords for each intent
    intent_map = {
        "SIDE_EFFECTS": ["side effect", "complication", "what effects", "suddenly stopping"],
        "DRUG_INTERACTION": ["interaction", "combination of drugs", "can i use", "after using", "with", "mix"],
        "ALCOHOL_INTERACTION": ["alcohol", "drinking", "beer", "wine"],
        "FOOD_INTERACTION": ["food", "diet", "ketogenic", "eat", "grapefruit"],
        "USES_INDICATIONS": ["used for", "how to treat", "treatment for", "treatment options", "what can be done", "how to manage", "interventions"],
        "STEWARDSHIP_MISSED": ["don't take", "missed", "forgot", "if i stop"],
        "REDIRECT_MEDICINE_QUERY": ["cause", "prevent", "natural", "heal", "how long", "why do i", "symptoms", "outcome", "test to rule out"],
        "MEDICAL_DEFINITION": ["what is", "what does", "mean", "definition", "mutation in"],
        "STEWARDSHIP_ADHERENCE": ["how long after", "keep dropping", "finish the course"],
        "REDIRECT_DOSAGE_QUERY": ["dosage", "mg", "dose", "how much", "how many"],
        "WARNING_PRECAUTIONS": ["warning", "caution", "safe to", "danger"],
        "ANTIBIOTIC_INFO": ["antibiotic", "penicillin", "amoxicillin"],
        "ABOUT_CHATBOT": ["who are you", "what can you do"],
        "DICTIONARY_LOOKUP": ["spelling", "pronounce"]
    }

    # Check for keyword matches
    for intent, keywords in intent_map.items():
        if any(keyword in query for keyword in keywords):
            return intent
            
    return "NOT_RECOGNIZED"

# Example usage with your provided list
questions = [
    "What are side effects of using formoterol?",
    "what is tamsulosin used for?",
    "can i use Sudafed after using Afrin for nasal congestion?",
    "what does cad without angina mean?"
]

for q in questions:
    print(f"Query: {q}\nIntent: {classify_medical_intent(q)}\n")

Query: What are side effects of using formoterol?
Intent: SIDE_EFFECTS

Query: what is tamsulosin used for?
Intent: USES_INDICATIONS

Query: can i use Sudafed after using Afrin for nasal congestion?
Intent: DRUG_INTERACTION

Query: what does cad without angina mean?
Intent: DRUG_INTERACTION



Medical Info 2019

In [12]:
pip install pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import pandas as pd
import re
import random
from collections import defaultdict

# Load dataset
med_info_raw_data = pd.read_csv("datasets/MedInfo2019-QA-Medications.csv")

# Drop unnecessary columns
med_info_raw_data.drop(['URL', 'Section Title', 'Answer'], axis=1, inplace=True)

# Antibiotics List
antibiotics = [
    "Amoxicillin",
    "Clarithromycin",
    "Azithromycin",
    "Doxycycline",
    "Cefuroxime",
    "Levofloxacin",
    "Cefixime",
    "Cefotaxime",
    "Fosfomycin",
    "Nitrofurantoin",
    "Trimethoprim-sulfamethoxazole",
    "Ciprofloxacin"
]

print(f"Original dataset size: {len(med_info_raw_data)}")
print(f"Target antibiotics: {len(antibiotics)}")

# Create a normalized version for comparison (lowercase)
target_drugs_lower = [drug.lower() for drug in antibiotics]

# Identify your columns
question_col = 'Question'  # Your question column name
drug_col = 'Focus (Drug)'  # Your drug column name

random.seed(42)

# Track counts to maintain balance
replacement_counts = defaultdict(int)

def map_to_target_drug_balanced(original_drug):
    """Map unknown drugs to maintain balanced distribution"""
    original_lower = str(original_drug).lower().strip()
    
    # Find which target drug has been used least for replacements
    if not replacement_counts:
        # If no replacements yet, initialize all counts to 0
        for drug in antibiotics:
            replacement_counts[drug] = 0
    
    min_count = min(replacement_counts.values())
    least_used = [drug for drug in antibiotics if replacement_counts[drug] == min_count]
    
    # Randomly choose among the least used
    chosen = random.choice(least_used)
    replacement_counts[chosen] += 1
    
    return chosen

# Transform the dataset
transformed_rows = []
stats = {
    'kept_original': 0,
    'replaced': 0,
    'skipped': 0,
    'skipped_terms': 0
}

for idx, row in med_info_raw_data.iterrows():
    # Get the original values
    original_question = str(row[question_col])
    original_drug = str(row[drug_col]).strip()
    
    # Skip if drug is empty
    if not original_drug or original_drug.lower() == 'nan':
        stats['skipped'] += 1
        continue
    
    # Check if the drug is already in your target list
    original_drug_lower = original_drug.lower()
    
    # Find if this drug matches any in your target list
    is_in_target = False
    matched_target = None
    
    # First, check for exact matches
    if original_drug_lower in target_drugs_lower:
        is_in_target = True
        matched_target = antibiotics[target_drugs_lower.index(original_drug_lower)]
    else:
        # Check if drug name contains any target drug or vice versa
        for target_drug, target_lower in zip(antibiotics, target_drugs_lower):
            if target_lower in original_drug_lower or original_drug_lower in target_lower:
                is_in_target = True
                matched_target = target_drug
                break
    
    if is_in_target:
        # Drug is already in your list - KEEP IT
        if original_drug != matched_target:
            # Replace with correct capitalization if needed
            pattern = re.compile(re.escape(original_drug), re.IGNORECASE)
            new_question = pattern.sub(matched_target, original_question)
            new_drug = matched_target
        else:
            # Already perfect, keep as is
            new_question = original_question
            new_drug = original_drug
        stats['kept_original'] += 1
    else:
        # Drug is NOT in your list - REPLACE IT with balanced random
        target_drug = map_to_target_drug_balanced(original_drug)
        
        # Check if drug was skipped due to skip terms
        if target_drug is None:
            stats['skipped_terms'] += 1
            continue
        
        # Replace the drug name in the question
        pattern = re.compile(re.escape(original_drug), re.IGNORECASE)
        new_question = pattern.sub(target_drug, original_question)
        new_drug = target_drug
        stats['replaced'] += 1
    
    # Create transformed row with original drug for reference
    new_row = row.to_dict()
    new_row[question_col] = new_question
    new_row[drug_col] = new_drug
    new_row['original_drug'] = original_drug  # Keep for reference
    
    transformed_rows.append(new_row)

# Create new dataframe with all transformed rows
new_df = pd.DataFrame(transformed_rows)

# Print detailed statistics
print("\n" + "="*60)
print("TRANSFORMATION STATISTICS")
print("="*60)
print(f"Total rows processed: {len(med_info_raw_data)}")
print(f"✅ Kept original drugs (already in antibiotic list): {stats['kept_original']}")
print(f"🔄 Replaced drugs (not in antibiotic list): {stats['replaced']}")
print(f"⚠️ Skipped (empty drug): {stats['skipped']}")
print(f"⚠️ Skipped (unwanted terms like marijuana, etc): {stats['skipped_terms']}")
print(f"\nFinal dataset size: {len(new_df)}")

# Show distribution of drugs in the transformed dataset
print("\n" + "="*60)
print("DRUG DISTRIBUTION IN TRANSFORMED DATASET")
print("="*60)
drug_counts = new_df[drug_col].value_counts()
for drug, count in drug_counts.items():
    percentage = (count / len(new_df)) * 100
    print(f"{drug}: {count} ({percentage:.1f}%)")

# Show how many of each original drug type
print("\n" + "="*60)
print("ORIGINAL VS TRANSFORMED SUMMARY")
print("="*60)
original_antibiotic_count = stats['kept_original']
replaced_count = stats['replaced']
print(f"Rows that originally had antibiotics: {original_antibiotic_count}")
print(f"Rows that originally had other drugs (now replaced): {replaced_count}")

# Show replacement distribution
if stats['replaced'] > 0:
    print("\n" + "="*60)
    print("REPLACEMENT DISTRIBUTION (How unknown drugs mapped to antibiotics)")
    print("="*60)
    for drug, count in replacement_counts.items():
        percentage = (count / stats['replaced']) * 100 if stats['replaced'] > 0 else 0
        print(f"{drug}: {count} ({percentage:.1f}%)")

# Show samples
print("\n" + "="*60)
print("SAMPLE TRANSFORMATIONS")
print("="*60)

# Show examples of kept drugs (originally antibiotics)
kept_samples = new_df[new_df['original_drug'] == new_df[drug_col]].head(3)
if len(kept_samples) > 0:
    print("\n✅ KEPT (drug was already an antibiotic):")
    for _, row in kept_samples.iterrows():
        print(f"  Original: {row['original_drug']} → Kept as: {row[drug_col]}")
        print(f"  Question: {row[question_col][:80]}...")
        print()

# Show examples of replaced drugs (originally non-antibiotics)
replaced_samples = new_df[new_df['original_drug'] != new_df[drug_col]].head(5)
if len(replaced_samples) > 0:
    print("\n🔄 REPLACED (drug was NOT an antibiotic):")
    for _, row in replaced_samples.iterrows():
        print(f"  Original: {row['original_drug']} → Replaced with: {row[drug_col]}")
        print(f"  Question: {row[question_col][:80]}...")
        print()

# Save transformed dataset
output_file = 'medinfo_questions.csv'
new_df.to_csv(output_file, index=False)
print(f"\n✅ Transformed dataset saved to '{output_file}'")

# Show a few original non-antibiotic drugs that got replaced
print("\n" + "="*60)
print("EXAMPLES OF ORIGINAL NON-ANTIBIOTIC DRUGS THAT WERE REPLACED")
print("="*60)
original_non_ab = new_df[new_df['original_drug'] != new_df[drug_col]]['original_drug'].value_counts().head(10)
for drug, count in original_non_ab.items():
    print(f"  {drug}: {count} occurrences")

Original dataset size: 690
Target antibiotics: 12

TRANSFORMATION STATISTICS
Total rows processed: 690
✅ Kept original drugs (already in antibiotic list): 16
🔄 Replaced drugs (not in antibiotic list): 673
⚠️ Skipped (empty drug): 1
⚠️ Skipped (unwanted terms like marijuana, etc): 0

Final dataset size: 689

DRUG DISTRIBUTION IN TRANSFORMED DATASET
Ciprofloxacin: 61 (8.9%)
Amoxicillin: 59 (8.6%)
Doxycycline: 58 (8.4%)
Azithromycin: 58 (8.4%)
Nitrofurantoin: 58 (8.4%)
Levofloxacin: 57 (8.3%)
Cefuroxime: 57 (8.3%)
Fosfomycin: 57 (8.3%)
Trimethoprim-sulfamethoxazole: 56 (8.1%)
Clarithromycin: 56 (8.1%)
Cefixime: 56 (8.1%)
Cefotaxime: 56 (8.1%)

ORIGINAL VS TRANSFORMED SUMMARY
Rows that originally had antibiotics: 16
Rows that originally had other drugs (now replaced): 673

REPLACEMENT DISTRIBUTION (How unknown drugs mapped to antibiotics)
Amoxicillin: 56 (8.3%)
Clarithromycin: 56 (8.3%)
Azithromycin: 56 (8.3%)
Doxycycline: 56 (8.3%)
Cefuroxime: 56 (8.3%)
Levofloxacin: 56 (8.3%)
Cefixime: 5

MEDQUAD

Web MD (Scraping)

In [ ]:
!pip install requests_html

In [ ]:
!pip install lxml_html_clean

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

# Your working headers
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}

# List of URLs to scrape
urls_to_scrape = [
    {'name': 'amoxicillin', 'url': 'https://www.webmd.com/drugs/amoxicillin-amoxil'},
    {'name': 'azithromycin', 'url': 'https://www.webmd.com/drugs/azithromycin-zithromax'},
    {'name': 'clarithromycin', 'url': 'https://www.webmd.com/drugs/clarithromycin-biaxin'},
    {'name': 'cefuroxime', 'url': 'https://www.webmd.com/drugs/cefuroxime-ceftin'},
    {'name': 'doxycycline', 'url': 'https://www.webmd.com/drugs/doxycycline-vibramycin'},
    {'name': 'levofloxacin', 'url': 'https://www.webmd.com/drugs/levofloxacin-levaquin'},
    {'name': 'cefixime', 'url': 'https://www.webmd.com/drugs/cefixime-suprax'},
    {'name': 'fosfomycin', 'url': 'https://www.webmd.com/drugs/fosfomycin-monurol'},
    {'name': 'nitrofurantoin', 'url': 'https://www.webmd.com/drugs/nitrofurantoin-macrobid-macrodantin'},  # Fixed spelling
    {'name': 'sulfamethoxazole and trimethoprim', 'url': 'https://www.webmd.com/drugs/sulfamethoxazole-trimethoprim-bactrim'},
    {'name': 'ciprofloxacin', 'url': 'https://www.webmd.com/drugs/ciprofloxacin-cipro'}
]

# Create an empty list to store all rows
all_rows = []

# Loop through each URL
for drug in urls_to_scrape:
    drug_name = drug['name']
    url = drug['url']

    print(f"\n{'='*50}")
    print(f"Scraping: {drug_name}")
    print(f"URL: {url}")
    print('='*50)

    # Make request
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Check if page loaded correctly

    # Find headings
    titles = soup.select("div.monograph-page h3")
    print(f"Found {len(titles)} h3 elements")

    # Store each heading as a separate row
    for title in titles:
        heading_text = title.get_text(strip=True)
        if heading_text:  # Only add if there's text
            all_rows.append({
                'drug': drug_name,
                'url': url,
                'heading': heading_text
            })

    # Be polite - wait between requests
    if drug != urls_to_scrape[-1]:  # Don't wait after last item
        time.sleep(2)

# Create DataFrame from all rows
web_md_scraped_df = pd.DataFrame(all_rows)

# Save to CSV
web_md_scraped_df.to_csv("webmd_scraped.csv", index=False, encoding='utf-8')